![](/Workspace/Users/sunnygupta2508@gmail.com/Databricks-Certified-Data-Engineer-Pro/Includes/images/customers.png)

In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql
select value::string
from bronze 
where topic = 'customers'

In [0]:
from pyspark.sql import functions as F

schema = "customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country_code STRING, row_status STRING, row_time timestamp"

customer_df = (spark.table("bronze")
                        .filter("topic = 'customers'")
                        .select(F.from_json(F.col("value").cast("string"), schema).alias("value"))
                        .select("value.*")
                        .filter(F.col("row_status").isin(["insert", "update"]))
               )
display(customer_df.orderBy("customer_id"))

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

ranked_df = (
    customer_df.withColumn("rank", F.rank().over(window))
                .filter(F.col("rank") == 1)
                .drop("rank")
)
display(ranked_df.orderBy("customer_id"))

In [0]:
from pyspark.sql.window import Window

def batch_upsert(microBatchDF, batchId):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

    (microBatchDF.filter(F.col("row_status").isin(["insert","update"]))
                 .withColumn("rank", F.rank().over(window))
                 .filter(F.col("rank") == 1)
                 .drop("rank")
                 .createOrReplaceTempView("ranked_updates")
    ) 

    query = """
        merge into customers_silver a
        using ranked_updates b
        on a.customer_id = b.customer_id
        when matched and a.row_time < b.row_time then update set *
        when not matched then insert *
    """
    microBatchDF.sparkSession.sql(query)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS customers_silver
(customer_id STRING, email STRING, first_name STRING, last_name STRING, gender STRING, street STRING, city STRING, country STRING, row_time TIMESTAMP)

In [0]:
df_country_lookup = spark.read.json(f"{bookstore.dataset_path}/country_lookup")
display(df_country_lookup)

In [0]:
query = (
    spark.readStream
            .table("bronze")
            .filter("topic = 'customers'")
            .select(F.from_json(F.col("value").cast("string"), schema).alias("value"))
            .select("value.*")
            .join(F.broadcast(df_country_lookup), F.col("country_code") == F.col("code"), "inner")
        .writeStream
            .foreachBatch(batch_upsert)
            .option("checkpointLocation", f"{bookstore.checkpoint_path}/customers_silver")
            .trigger(availableNow=True)
            .start()
    
)
query.awaitTermination()

In [0]:
%sql
select *
from customers_silver

In [0]:
count = (spark.read.table("customers_silver").count())
expected_count = (spark.read.table("customers_silver").select("customer_id").distinct().count())

assert count == expected_count, "The number of records in the customers_silver table is not correct"
print("Unit test passed")